In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/restricted/kidney_waitlist_analytic.csv.gz")

/tmp/ipykernel_693015/3652200463.py:1: DtypeWarning: Columns (22: PREV_TX, 23: TX_DATE, 24: DON_TY, 26: MULTIORG, 27: ORGAN, 30: TRR_ID_CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/restricted/kidney_waitlist_analytic.csv.gz")


In [3]:
# find and sum all null columns
df.isna().sum()

ON_DIALYSIS                  0
A2A2B_ELIGIBILITY       456604
GENDER                       0
ABO                          0
BMI_TCR                   1558
FUNC_STAT_TCR             3975
INIT_STAT                    0
INIT_CPRA               142741
END_CPRA                142448
REM_CD                  103300
DAYSWAIT_CHRON              53
END_STAT                     0
INIT_AGE                     0
DIALYSIS_DATE           134245
END_DATE                     0
INIT_DATE                    0
ETHCAT                       0
PT_CODE                      0
DAYSWAIT_ALLOC            6745
COMPOSITE_DEATH_DATE    398512
REGION                       0
WL_ID_CODE                   0
PREV_TX                 264107
TX_DATE                 264107
DON_TY                  264107
DIAG_KI                 265540
MULTIORG                483004
ORGAN                   264107
PSTATUS                 264107
PTIME                   267350
TRR_ID_CODE             264107
DONOR_ID                264107
LISTING_

In [4]:
# see the data type of each column
df.dtypes

ON_DIALYSIS                 str
A2A2B_ELIGIBILITY       float64
GENDER                      str
ABO                         str
BMI_TCR                 float64
FUNC_STAT_TCR           float64
INIT_STAT                 int64
INIT_CPRA               float64
END_CPRA                float64
REM_CD                  float64
DAYSWAIT_CHRON          float64
END_STAT                  int64
INIT_AGE                  int64
DIALYSIS_DATE               str
END_DATE                    str
INIT_DATE                   str
ETHCAT                    int64
PT_CODE                   int64
DAYSWAIT_ALLOC          float64
COMPOSITE_DEATH_DATE        str
REGION                    int64
WL_ID_CODE                int64
PREV_TX                     str
TX_DATE                     str
DON_TY                      str
DIAG_KI                 float64
MULTIORG                    str
ORGAN                       str
PSTATUS                 float64
PTIME                   float64
TRR_ID_CODE                 str
DONOR_ID

In [5]:
# drop columns that tell the model the answer
# TX_DATE, DON_TY, DONOR_ID, ORGAN, TRR_ID_CODE only get populated when a transplant happens
# REM_CD, END_DATE, END_STAT, END_CPRA describe status after leaving waitlist
# COMPOSITE_DEATH_DATE only has a val if the patient dies
# DAYSWAIT_CHRON, DAYSWAIT_ALLOC same nums as the ones we want to predict
# PTIME, PSTATUS post transplant
# Basically, everything populated after the transplant happens has to be removed
drop_col = [
    'REM_CD', 'END_DATE', 'TX_DATE', 'COMPOSITE_DEATH_DATE',
    'END_STAT', 'END_CPRA', 'DAYSWAIT_CHRON', 'DAYSWAIT_ALLOC',
    'DON_TY', 'ORGAN', 'TRR_ID_CODE', 'DONOR_ID', 'PTIME', 'PSTATUS'
]
# safety check for columns that don't exist. if the value exists, drop it
df_features = df.drop(columns=[col for col in drop_col if col in df.columns])

dropped_columns = [col for col in drop_col if col in df_features.columns]
if(dropped_columns): # if this is true, columns were not dropped properly
    print("Columns were not dropped properly!")
else:
    print("Columns don't exist anymore.")

Columns don't exist anymore.


In [6]:
# fill string-type columns with missing values
# fill with N/A, which is more accurate than unknown
df_features['A2A2B_ELIGIBILITY'] = df_features['A2A2B_ELIGIBILITY'].fillna("N/A")

# fill dialysis date with "not on dialysis" because unknown is wrong
df_features['DIALYSIS_DATE'] = df_features['DIALYSIS_DATE'].fillna('Not on dialysis')

# replace missing values with with "N"
df_features['MULTIORG'] = df_features['MULTIORG'].fillna('N')
df['DIAG_KI'].value_counts(dropna=False).head(20)

string_cols_list = df_features.select_dtypes(include=['str', 'object']).columns.tolist()
df_features[string_cols_list] = df_features[string_cols_list].fillna("Unknown") # we can just put unknown here

#print all string values with null values. if it's 0, it means all null values have been accounted for
df_features.select_dtypes(include=['object', 'string']).isna().sum()

ON_DIALYSIS          0
A2A2B_ELIGIBILITY    0
GENDER               0
ABO                  0
DIALYSIS_DATE        0
INIT_DATE            0
PREV_TX              0
MULTIORG             0
outcome              0
dtype: int64

In [7]:
# fill numeric (int and float) type columns with missing values

# continuous fields: fill them with the median. don't put dummy values like 0, which would corrupt the data. distribution likely skewed, so use median over mean
continuous_num = ['BMI_TCR', 'INIT_CPRA']
df_features[continuous_num] = df_features[continuous_num].fillna(df_features[continuous_num].median())

print(df_features['BMI_TCR'].isna().sum())
print(df_features['INIT_CPRA'].isna().sum())

# fill DIAG_KI with a sentinel that's an invalid number
df_features['DIAG_KI'] = df_features['DIAG_KI'].fillna(-1)
# convert it to a string object, so that it's not treated as a number
df_features['DIAG_KI'] = df_features['DIAG_KI'].astype('str')

# fill FUNC_STAT_TCR with a sentinel that's an invalid number
df_features['FUNC_STAT_TCR'] = df_features['FUNC_STAT_TCR'].fillna(-1)
# convert it to a string object, so that it's not treated as a number
df_features['FUNC_STAT_TCR'] = df_features['FUNC_STAT_TCR'].astype('str')


print(df_features['DIALYSIS_DATE'].isna().sum())
print(df_features['MULTIORG'].isna().sum())
print(df_features['DIAG_KI'].isna().sum())

0
0
0
0
0


In [8]:
# exclude bad dates in days_to_event
df_features = df_features.loc[~(df['END_DATE'] < df['INIT_DATE'])].copy()

In [9]:
# make sure they're all null
df_features.isna().sum()

ON_DIALYSIS          0
A2A2B_ELIGIBILITY    0
GENDER               0
ABO                  0
BMI_TCR              0
FUNC_STAT_TCR        0
INIT_STAT            0
INIT_CPRA            0
INIT_AGE             0
DIALYSIS_DATE        0
INIT_DATE            0
ETHCAT               0
PT_CODE              0
REGION               0
WL_ID_CODE           0
PREV_TX              0
DIAG_KI              0
MULTIORG             0
LISTING_CTR_CODE     0
outcome              0
event_adverse        0
event_transplant     0
censored             0
days_to_event        0
dtype: int64

In [10]:
# find outliers in numeric-type columns and winsorize using IQR
to_winscorize = ['BMI_TCR']
# INIT_CPRA reacted badly to IQR, so leave it alone. it's a percentage, so the valeus cannot really be outliers
# INIT_AGE also reacted badly (cut away valid ages)
# DON'T WINSCORIZE DAYS_TO_EVENT--CUTTING AWAY OUTLEIRS CUTS OUT DATA!!!

# IQR (retains data size and reduces distortion)
for col in to_winscorize:
    Q1 = df_features[col].quantile(0.25)
    Q3 = df_features[col].quantile(0.75)
    IQR = Q3-Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_features[(df_features[col] < lower_bound) | (df_features[col] > upper_bound)][col]
    print(f"{col}: {len(outliers)} outliers, range [{outliers.min()}, {outliers.max()}], bounds [{lower_bound:.1f}, {upper_bound:.1f}]")

BMI_TCR: 2652 outliers, range [0.18, 430226.67], bounds [12.4, 45.1]
